# E2.2 · Horizontal AI regulation

**Function E — AI Governance for Agentic Systems → Building the Governance Platform — Regulatory and Compliance**  ·  *Security of AI*

Builds on **[E2.1 · The regulatory map](https://spbreed.github.io/cyber-commons/lessons/E2.1.html)**.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Horizontal AI regulation applies to you regardless of sector, and its obligations are structural: risk management, documentation, oversight. Those are programme requirements, not paperwork requirements.

> **At CyberTravels.** Horizontal AI obligations land on TripBot as programme requirements: a risk register, technical documentation, named human oversight, measured accuracy, post-market monitoring.

## 2 · The framework

```
   horizontal obligations are structural, not clerical

   risk management system     -> you need a register and a tiering model
   technical documentation    -> generated, not written once
   human oversight            -> a named person with real authority
   accuracy / robustness      -> measured, with the method recorded
   post-market monitoring     -> drift detection, by another name
```

Horizontal AI regulation is, in practice, mostly about **documented process and
human oversight**. That is good news, because those map onto controls you can
build and evidence mechanically.

The trap is answering a clause with a policy document. "We maintain appropriate
human oversight" satisfies nobody who asks the follow-up question, and the
follow-up question is always the same: *show me*.

So the working method is to resolve each regulatory theme down to a control from
your own catalogue (E1.4), and let the control's evidence be the answer. Four
themes cover most of it:

- risk management system,
- record-keeping,
- human oversight,
- accuracy and robustness.

Each one resolves to controls you already built in tracks A, B and D.

## 3 · Demo — resolve each theme to a control with evidence

In [ ]:
CATALOGUE = {
 "AC-1": ("agent identities distinct from human and separately revocable",
          "gateway logs with an act chain; monthly sample"),
 "AC-2": ("delegated authority narrows at every hop",
          "regression suite IDN-01/IDN-04 on every release"),
 "SB-1": ("egress deny-by-default with an allowlist", "90-day denial log"),
 "SB-2": ("privileged tools require approval below L3", "tool policy in git + denial log"),
 "EV-1": ("every action logged with the acting identity", "audit sample of 50 actions"),
 "EV-2": ("accuracy evaluated against a held-out key per release",
          "expert accuracy report with sample size"),
 "DR-1": ("behavioural drift raises an alert", "drift alerts and dispositions"),
 "ST-1": ("a tested stop mechanism you own", "game-day record with measured time-to-stop"),
}
THEMES = {
 "risk management system":       ["AC-1", "SB-2", "DR-1"],
 "record-keeping (Art.12)":      ["EV-1", "AC-2"],
 "human oversight (Art.14)":     ["SB-2", "ST-1"],
 "accuracy and robustness":      ["EV-2", "DR-1"],
}
for theme, cids in THEMES.items():
    print(f"{theme}")
    for cid in cids:
        text, evidence = CATALOGUE[cid]
        print(f"   {cid}  {text}")
        print(f"         evidence: {evidence}")
    print()

## 4 · Where it breaks — the clause answered with prose

In [ ]:
WEAK = {
 "risk management system":   "We operate a risk management framework for AI systems.",
 "record-keeping (Art.12)":  "Appropriate logs are retained.",
 "human oversight (Art.14)": "Human oversight is maintained at all times.",
 "accuracy and robustness":  "Models are tested prior to deployment.",
}
def survives_followup(answer, controls):
    """The follow-up question is always 'show me'."""
    return bool(controls), ("names a control with an artefact" if controls
                            else "no artefact — the answer IS the evidence, which is the problem")

print(f"{'theme':28s}{'prose answer survives?':>24}")
print("-" * 56)
for theme in THEMES:
    ok_weak, _ = survives_followup(WEAK[theme], [])
    print(f"{theme:28s}{str(ok_weak):>24}")
print("\nAll four fail the same way: there is nothing to produce when asked.")

## 5 · The control — produce the evidence, then check it is fresh

In [ ]:
import time
from dataclasses import dataclass
now = time.time(); DAY = 86400

@dataclass
class ControlTest:
    cid: str; passed: bool; tested_at: float; valid_for_days: float
    def state(self, at):
        if (at - self.tested_at)/DAY > self.valid_for_days: return "STALE"
        return "PASS" if self.passed else "FAIL"

TESTS = [ControlTest("AC-1", True,  now -  3*DAY, 30),
         ControlTest("AC-2", True,  now -  9*DAY, 30),
         ControlTest("SB-2", True,  now - 40*DAY, 30),
         ControlTest("EV-1", True,  now -  5*DAY, 60),
         ControlTest("EV-2", True,  now - 12*DAY, 30),
         ControlTest("DR-1", False, now,          30),
         ControlTest("ST-1", True,  now - 41*DAY, 180)]
by = {t.cid: t for t in TESTS}

print(f"{'theme':28s}{'controls':22s}{'evidenced now':>15}")
print("-" * 68)
for theme, cids in THEMES.items():
    states = [by[c].state(now) if c in by else "NO EVIDENCE" for c in cids]
    ok = all(s == "PASS" for s in states)
    blockers = ",".join(c for c, s in zip(cids, states) if s != "PASS")
    verdict = "yes" if ok else f"NO — {blockers}"
    print(f"{theme:28s}{str(cids):22s}{verdict:>15}")

fully = [t for t, cids in THEMES.items()
         if all((by[c].state(now) if c in by else "X") == "PASS" for c in cids)]
print(f"\nthemes fully evidenced right now: {len(fully)}/{len(THEMES)}  {fully}")
print("\nThat sentence is what you say to a supervisor. It is smaller than the")
print("prose version and it is defensible, which is the trade worth making.")
assert len(fully) < len(THEMES)

## What you just proved

Four regulatory themes resolve to named controls, each with a concrete evidence artefact. All four prose answers fail the show-me test. Checking freshness, two themes are fully evidenced — human oversight fails on a stale SB-2 and risk management on a failing DR-1 — giving a smaller but defensible statement.

## Your turn

Take one clause your programme claims to satisfy and trace it to an artefact with a date. If the trail ends at a policy document, the clause is ticked and undefended.

---

**Next → [E2.3 · Voluntary frameworks as your spine](https://spbreed.github.io/cyber-commons/lessons/E2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*